# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 03: Feature Engineering
# ==========================================================

"""
Objective
---------
1. Load the cleaned dataset generated in Notebook 02.
2. Create the target variable.
3. Select predictive features.
4. Construct derived clinical features.
5. Validate feature quality and consistency.
6. Build the final feature dataset.
7. Export the feature dataset in Parquet format.

Mục tiêu
---------
1. Nạp dữ liệu đã làm sạch từ Notebook 02.
2. Xây dựng biến mục tiêu.
3. Lựa chọn các thuộc tính phục vụ dự đoán.
4. Tạo các đặc trưng lâm sàng mới.
5. Kiểm tra tính nhất quán của các đặc trưng.
6. Hoàn thiện tập dữ liệu đặc trưng.
7. Xuất tập dữ liệu đặc trưng dưới định dạng Parquet.
"""

In [1]:
# 1. Import Libraries | Khai báo thư viện

print("=" * 60)
print("1. IMPORT LIBRARIES")
print("=" * 60)

import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Add project root directory to path | Thêm thư mục gốc dự án vào hệ thống
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions from src | Nạp các hàm tự định nghĩa từ src
from src.data.loader import create_spark_session, load_csv

1. IMPORT LIBRARIES


In [2]:
# 2. Create Spark Session | Khởi tạo Spark Session

print("=" * 60)
print("2. CREATE SPARK SESSION")
print("=" * 60)

spark = create_spark_session("SEER Breast Cancer Feature Engineering")

2. CREATE SPARK SESSION


In [3]:
# 3. Load Clean Dataset | Tải dữ liệu đã làm sạch

print("=" * 60)
print("3. LOAD CLEANED DATASET")
print("=" * 60)

df = spark.read.parquet("../data/processed/seer_breast_cancer_clean")

print(f"Initial Rows    : {df.count():,}")
print(f"Initial Columns : {len(df.columns)}")

3. LOAD CLEANED DATASET
Initial Rows    : 456,087
Initial Columns : 24


In [4]:
# 4. Define Target Variable | Xây dựng biến mục tiêu

print("=" * 60)
print("4. DEFINE TARGET VARIABLE")
print("=" * 60)

# Transform Vital_Status: Alive -> 0, Dead -> 1 (Positive Class) | Chuyển đổi nhãn sinh tử về dạng nhị phân
df = df.withColumn(
    "label",
    F.when(F.col("Vital_Status") == "Alive", 0)
    .when(F.col("Vital_Status") == "Dead", 1)
    .otherwise(None)
)

# Remove original target column to prevent redundancy | Loại bỏ cột nhãn gốc
df = df.drop("Vital_Status")

# Verify label distribution | Kiểm soát tỷ lệ phân phối lớp mục tiêu
print("Target Variable (label) Distribution:")
df.groupBy("label").count().show()

4. DEFINE TARGET VARIABLE
Target Variable (label) Distribution:
+-----+------+
|label| count|
+-----+------+
|    1|168781|
|    0|287306|
+-----+------+



In [5]:
# 5. Create Age Group Feature | Tạo đặc trưng nhóm tuổi

print("=" * 60)
print("5. AGE GROUP EXTRACTION")
print("=" * 60)

# Categorize patient age into 4 clinical groups | Chia khoảng tuổi bệnh nhân thành 4 lớp lâm sàng
df = df.withColumn(
    "Age_Group",
    F.when(F.col("Age") < 40, "Under_40")
    .when((F.col("Age") >= 40) & (F.col("Age") < 60), "40_to_59")
    .when((F.col("Age") >= 60) & (F.col("Age") < 80), "60_to_79")
    .otherwise("80_and_Above")
)

print("Age Group Distribution:")
df.groupBy("Age_Group").count().show()

5. AGE GROUP EXTRACTION
Age Group Distribution:
+------------+------+
|   Age_Group| count|
+------------+------+
|80_and_Above|456087|
+------------+------+



In [6]:
# 6. Create Tumor Size Group Feature | Tạo đặc trưng nhóm kích thước khối u

print("=" * 60)
print("6. TUMOR SIZE GROUP EXTRACTION")
print("=" * 60)

# Categorize tumor size according to AJCC guidelines | Phân nhóm kích thước u theo chuẩn lâm sàng AJCC
df = df.withColumn(
    "Tumor_Size_Group",
    F.when(F.col("Tumor_Size") <= 20, "T1_Micro")
    .when((F.col("Tumor_Size") > 20) & (F.col("Tumor_Size") <= 50), "T2_Medium")
    .when(F.col("Tumor_Size") > 50, "T3_Large")
    .otherwise("Unknown")
)

print("Tumor Size Group Distribution:")
df.groupBy("Tumor_Size_Group").count().show()

6. TUMOR SIZE GROUP EXTRACTION
Tumor Size Group Distribution:
+----------------+------+
|Tumor_Size_Group| count|
+----------------+------+
|         Unknown| 37004|
|        T1_Micro|243452|
|       T2_Medium|140029|
|        T3_Large| 35602|
+----------------+------+



In [7]:
# 7. Create Node Ratio Feature | Tạo đặc trưng tỷ lệ hạch

print("=" * 60)
print("7. NODE INVOLVEMENT RATIO")
print("=" * 60)

# Define Node Ratio: positive nodes / examined nodes | Công thức: Số hạch dương tính / Số hạch khảo sát
df = df.withColumn(
    "Node_Ratio",
    F.when(
        (F.col("Regional_Nodes_Examined") > 0) & (F.col("Regional_Nodes_Positive").isNotNull()),
        F.col("Regional_Nodes_Positive") / F.col("Regional_Nodes_Examined")
    ).otherwise(0.0)
)

# Cap node ratio logically at 1.0 | Khống chế cận trên của tỷ lệ hạch ở mức tối đa 1.0
df = df.withColumn(
    "Node_Ratio",
    F.when(F.col("Node_Ratio") > 1.0, 1.0).otherwise(F.col("Node_Ratio"))
)

df.select("Regional_Nodes_Positive", "Regional_Nodes_Examined", "Node_Ratio").show(10)

7. NODE INVOLVEMENT RATIO
+-----------------------+-----------------------+------------------+
|Regional_Nodes_Positive|Regional_Nodes_Examined|        Node_Ratio|
+-----------------------+-----------------------+------------------+
|                   NULL|                      0|               0.0|
|                      0|                     15|               0.0|
|                      0|                      4|               0.0|
|                   NULL|                      0|               0.0|
|                      0|                      1|               0.0|
|                      6|                     11|0.5454545454545454|
|                   NULL|                      0|               0.0|
|                      0|                     18|               0.0|
|                   NULL|                      0|               0.0|
|                      0|                      4|               0.0|
+-----------------------+-----------------------+------------------+
only sho

In [8]:
# 8. Create Hormone Status Feature | Tạo đặc trưng trạng thái hormone

print("=" * 60)
print("8. HORMONE RECEPTOR COMBINED STATUS")
print("=" * 60)

# Combine PR and ER Status into clinical categories | Tổ hợp hai chỉ số thụ thể ER và PR thành các nhánh lâm sàng
df = df.withColumn(
    "Hormone_Status",
    F.when((F.col("ER_Status") == "Positive") & (F.col("PR_Status") == "Positive"), "HR_Positive")
    .when((F.col("ER_Status") == "Negative") & (F.col("PR_Status") == "Negative"), "HR_Negative")
    .otherwise("HR_Mixed")
)

print("Hormone Combined Status Distribution:")
df.groupBy("Hormone_Status").count().show()

8. HORMONE RECEPTOR COMBINED STATUS
Hormone Combined Status Distribution:
+--------------+------+
|Hormone_Status| count|
+--------------+------+
|   HR_Negative| 77697|
|   HR_Positive|288396|
|      HR_Mixed| 89994|
+--------------+------+



In [9]:
# 9. Impute Numerical Missing Values | Điền giá trị khuyết cho biến số

print("=" * 60)
print("9. NULL IMPUTATION FOR NUMERICAL FEATURES")
print("=" * 60)

# Perform Median Imputation on key numerical attributes | Tính toán trung vị để điền rỗng thuộc tính số
numerical_features = ["Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive"]

for col in numerical_features:
    median_val = df.approxQuantile(col, [0.5], 0.01)[0]
    df = df.fillna({col: median_val})
    print(f"Column: {col:<25} | Imputed Nulls with Median: {median_val}")

9. NULL IMPUTATION FOR NUMERICAL FEATURES
Column: Tumor_Size                | Imputed Nulls with Median: 18.0
Column: Regional_Nodes_Examined   | Imputed Nulls with Median: 3.0
Column: Regional_Nodes_Positive   | Imputed Nulls with Median: 0.0


In [10]:
# 10. Convert Data Types | Chuyển đổi kiểu dữ liệu

print("=" * 60)
print("10. TYPE CASTING & ROBUST REGEX FOR AGE")
print("=" * 60)

# 1. Xử lý bóc tách số cho cột Age bằng RegEx trước khi Cast (Tránh phát sinh Null)
df = df.withColumn("Age_Cleaned", F.regexp_extract(F.col("Age"), r"(\d+)", 1))
df = df.withColumn("Age", F.col("Age_Cleaned").cast("double")).drop("Age_Cleaned")

# 2. Ép kiểu các thuộc tính danh mục phức tạp sang String
df = df.withColumn("Histologic_Type", F.col("Histologic_Type").cast("string"))
df = df.withColumn("Surgery_Primary_Site", F.col("Surgery_Primary_Site").cast("string"))

print("Data type casting and Age RegEx extraction finalized successfully.")

10. TYPE CASTING & ROBUST REGEX FOR AGE
Data type casting and Age RegEx extraction finalized successfully.


In [11]:
# 11. Remove Intermediate Features | Loại bỏ đặc trưng trung gian

print("=" * 60)
print("11. REMOVING INTERMEDIARY COLUMNS")
print("=" * 60)

intermediate_cols = ["ER_Status", "PR_Status"]
df = df.drop(*intermediate_cols)

print(f"Dropped redundant intermediary columns: {intermediate_cols}")

11. REMOVING INTERMEDIARY COLUMNS
Dropped redundant intermediary columns: ['ER_Status', 'PR_Status']


In [12]:
# 12. Validate Feature Dataset | Kiểm tra tập đặc trưng

print("=" * 60)
print("12. FEATURE DATASET VALIDATION")
print("=" * 60)

print(f"Final Features Rows    : {df.count():,}")
print(f"Final Features Columns : {len(df.columns)}")
df.printSchema()

# Ensure no NULLs remain in critical numerical features
critical_cols = ["Age", "Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive", "Node_Ratio"]
for column in critical_cols:
    null_count = df.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Alert: {column:30} has {null_count:,} nulls remaining!")
    else:
        print(f"Success: {column:30} is 100% clean.")

12. FEATURE DATASET VALIDATION
Final Features Rows    : 456,087
Final Features Columns : 26
root
 |-- Age: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: integer (nullable = true)
 |-- Grade: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: string (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: string (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = true)
 |-- Radiation: string (nullable = true)
 |-- Chemotherapy: string (nullable = true)
 |-- 

In [13]:
# 13. Export Feature Dataset | Xuất tập đặc trưng

import shutil
print("=" * 60)
print("13. EXPORT FEATURE DATASET")
print("=" * 60)

output_path = os.path.abspath("../data/processed/seer_breast_cancer_feature") 
if os.path.exists(output_path):
    shutil.rmtree(output_path)
    
print(f"Writing feature dataset to: {output_path}")

df.write.mode("overwrite").parquet(output_path)

print("Feature dataset successfully exported.")

13. EXPORT FEATURE DATASET
Writing feature dataset to: d:\Project_Breast_Cancer_SEER\data\processed\seer_breast_cancer_feature
Feature dataset successfully exported.
